# Tahap 4: Case Solution Reuse
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Gunakan putusan lama sebagai dasar prediksi solusi untuk kasus baru

### Alur Notebook:
1. Install & Import
2. Load Artefak dari Tahap 3
3. Fungsi Ekstraksi Solusi dari Top-K Kasus
4. Algoritma Prediksi (Majority Vote + Weighted Similarity)
5. Fungsi `predict_outcome()` Terpadu
6. Demo Manual 5 Kasus Baru
7. Simpan Hasil Prediksi ke `predictions.csv`

## Cell 2 — Import Library

In [1]:
import json
import re
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from getpass import getpass

import joblib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer


warnings.filterwarnings("ignore")
print("✅ Library berhasil diimport")

d:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\PK\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Library berhasil diimport


## Cell 3 — Konfigurasi

In [2]:
CONFIG = {
    "TOP_K"          : 3,     # 3 untuk dataset kecil, 5 untuk final
    "RETRIEVAL_MODE" : "tfidf",   # "tfidf" atau "embedding"
    "GROQ_MODEL"     : "llama-3.3-70b-versatile",
    "GROQ_DELAY"     : 1.5,
    "MAX_CHARS_SOLUSI": 3000,     # batas karakter teks solusi yang dikirim ke Groq
}

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

STEP1_DIR = PROJECT_ROOT / "STEP 1"
STEP3_DIR = PROJECT_ROOT / "STEP 3"

DATA_PROC   = STEP1_DIR / "data" / "processed"
DATA_EVAL   = STEP1_DIR / "data" / "eval"

DATA_RESULT = PROJECT_ROOT / "results"

MODEL_DIR   = STEP3_DIR / "models"
LOGS_DIR    = PROJECT_ROOT / "logs"

DATA_RESULT.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

DATA_RESULT.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(levelname)s | %(message)s",
    handlers = [
        logging.FileHandler(LOGS_DIR / "solution_reuse.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print("✅ Konfigurasi selesai")

✅ Konfigurasi selesai


## Cell 4 — Load Artefak dari Tahap 3

In [3]:
# ── Load cases (lengkap dengan teks_full) ──────────────────
json_path = DATA_PROC / "cases.json"
assert json_path.exists(), f"❌ {json_path} tidak ditemukan. Jalankan Tahap 2 dulu."

with open(json_path, encoding="utf-8") as f:
    cases = json.load(f)

df = pd.DataFrame(cases)
print(f"✅ {len(df)} kasus dimuat dari cases.json")

# ── Load TF-IDF ─────────────────────────────────────────────
tfidf    = joblib.load(MODEL_DIR / "tfidf_vectorizer.pkl")
X_tfidf  = joblib.load(MODEL_DIR / "tfidf_matrix.pkl")
print(f"✅ TF-IDF dimuat — matrix: {X_tfidf.shape}")

# ── Load SentenceTransformer embeddings ─────────────────────
embeddings = np.load(MODEL_DIR / "embeddings.npy")
print(f"✅ Embeddings dimuat — shape: {embeddings.shape}")

# ── Load SentenceTransformer model ──────────────────────────
ST_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"\n🔄 Loading SentenceTransformer model...")
st_model = SentenceTransformer(ST_MODEL)
print(f"✅ ST model siap")

# ── Buat dict solusi per case_id ─────────────────────────────
# Solusi = amar putusan + vonis (ini yang akan di-reuse)
case_solutions = {}
for _, row in df.iterrows():
    cid = row["case_id"]
    case_solutions[cid] = {
        "amar_putusan"   : str(row.get("amar_putusan", "")),
        "pasal_terbukti" : str(row.get("pasal_terbukti", "")),
        "vonis_penjara"  : str(row.get("vonis_penjara", "")),
        "vonis_denda"    : str(row.get("vonis_denda", "")),
        "ringkasan_fakta": str(row.get("ringkasan_fakta", "")),
    }

print(f"✅ {len(case_solutions)} solusi kasus tersedia")

2026-06-13 23:53:30,600 | INFO | No device provided, using cpu


✅ 43 kasus dimuat dari cases.json
✅ TF-IDF dimuat — matrix: (43, 3616)
✅ Embeddings dimuat — shape: (43, 384)

🔄 Loading SentenceTransformer model...


2026-06-13 23:53:31,027 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 23:53:31,056 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
2026-06-13 23:53:31,307 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 23:53:31,308 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-13 23:53:31,335 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc

✅ ST model siap
✅ 43 solusi kasus tersedia


In [4]:
print(df.columns.tolist())

['case_id', 'url_detail', 'path_txt', 'no_perkara', 'tanggal_putusan', 'pengadilan', 'terdakwa', 'jaksa', 'pasal_dakwaan', 'pasal_terbukti', 'amar_putusan', 'vonis_penjara', 'vonis_denda', 'barang_bukti', 'ringkasan_fakta', 'jumlah_kata', 'jumlah_kalimat', 'jumlah_paragraf', 'avg_kata_kalimat', 'kata_kunci_pa', 'n_kata_kunci_pa', 'sumber_ekstraksi', 'teks_full']


In [6]:
# ============================================================
# MODE TANPA GROQ — prediksi pakai majority vote + weighted similarity
# ============================================================
groq_client = None
GROQ_AKTIF  = False
print("⏭️  Groq dilewati — prediksi 100% rule-based (offline)")

⏭️  Groq dilewati — prediksi 100% rule-based (offline)


## Cell 6 — Fungsi Retrieve (diulang dari Tahap 3)

In [7]:
STOPWORDS_ID = set([
    "yang","dan","di","ke","dari","dengan","untuk","pada","dalam",
    "adalah","ini","itu","atau","juga","sudah","oleh","tidak","ada",
    "bahwa","telah","akan","maka","serta","para","kepada","karena",
    "atas","sebagai","dapat","tersebut","sehingga","antara","nomor",
    "halaman","republik","indonesia","mahkamah","pengadilan","agung",
])

def preprocess_teks(teks: str, max_chars: int = 8000) -> str:
    if not teks or not isinstance(teks, str): return ""
    teks = teks[:max_chars].lower()
    teks = re.sub(r"\d+", " ", teks)
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    kata = [k for k in teks.split() if k not in STOPWORDS_ID and len(k) > 2]
    return " ".join(kata)

def buat_teks_representasi(row) -> str:
    bagian = []
    for field in ["pasal_dakwaan","pasal_terbukti","amar_putusan"]:
        val = str(row.get(field,"")).strip()
        if val:
            bagian.append(val); bagian.append(val)
    for field in ["ringkasan_fakta","barang_bukti","terdakwa"]:
        val = str(row.get(field,"")).strip()
        if val: bagian.append(val)
    teks_penuh = str(row.get("teks_full",""))[:3000]
    if teks_penuh: bagian.append(teks_penuh)
    return " ".join(bagian)

def retrieve(query: str, k: int = None, mode: str = None) -> list:
    if k    is None: k    = CONFIG["TOP_K"]
    if mode is None: mode = CONFIG["RETRIEVAL_MODE"]

    query_clean = preprocess_teks(query)
    if not query_clean: return []

    if mode == "tfidf":
        q_vec  = tfidf.transform([query_clean])
        scores = cosine_similarity(q_vec, X_tfidf).flatten()
    elif mode == "embedding":
        q_emb  = st_model.encode([query], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, embeddings).flatten()
    else:
        raise ValueError(f"Mode tidak dikenal: {mode}")

    top_idx = np.argsort(scores)[::-1][:k]
    hasil = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[idx]
        hasil.append({
            "rank"           : rank,
            "case_id"        : row["case_id"],
            "score"          : round(float(scores[idx]), 4),
            "no_perkara"     : str(row.get("no_perkara","")),
            "pengadilan"     : str(row.get("pengadilan","")),
            "terdakwa"       : str(row.get("terdakwa","")),
            "pasal_dakwaan"  : str(row.get("pasal_dakwaan","")),
            "pasal_terbukti" : str(row.get("pasal_terbukti","")),
            "amar_putusan"   : str(row.get("amar_putusan","")),
            "vonis_penjara"  : str(row.get("vonis_penjara","")),
            "ringkasan_fakta": str(row.get("ringkasan_fakta","")),
        })
    return hasil

print("✅ Fungsi retrieve() siap")

✅ Fungsi retrieve() siap


In [8]:
hasil = retrieve(
    "persetubuhan anak usia 13 tahun",
    k=3
)

from pprint import pprint
pprint(hasil)

[{'amar_putusan': 'terbukti bersalah melakukan tindak pidana : “ Menculik anak '
                  'untuk diri sendiri” dalam pasal 83 UU RI No',
  'case_id': 'case_052',
  'no_perkara': '764/Pid.B/2014/PN.Dps.',
  'pasal_dakwaan': '',
  'pasal_terbukti': '',
  'pengadilan': 'Pengadilan Negeri Denpasar',
  'rank': 1,
  'ringkasan_fakta': 'hkama ahkamah Agung Repub ahkamah Agung Republik '
                     'Indonesia mah Agung Republik Indonesia blik Indonesi P U '
                     'T U S A N Nomor : 764/Pid.B/2014/PN.Dps. “DEMI KEADILAN '
                     'BERDASARKAN KETUHANAN YANG MAHA ESA” Pengadilan Negeri '
                     'Denpasar yang memeriksa dan mengadili perkara-perkara '
                     'pidana dalam peradilan tingka',
  'score': 0.0694,
  'terdakwa': 'yang pada pokoknya memohon keringanan hukuman pada Majelis '
              'hakim oleh karena ia telah menyesali perbuatannya dan berjanji '
              'tidak akan mengulangi lagi',
  'vonis_penjara'

## Cell 7 — Algoritma Prediksi Solusi

### Dua pendekatan sesuai spesifikasi tugas:
1. **Majority Vote** — pilih amar putusan yang paling sering muncul di top-k kasus
2. **Weighted Similarity** — ambil solusi dari kasus dengan skor similarity tertinggi

In [9]:
def majority_vote_solution(top_k):
    solusi = []

    for h in top_k:
        amar = case_solutions[h["case_id"]]["amar_putusan"]

        if amar:
            solusi.append(amar)

    if not solusi:
        return ""

    return Counter(solusi).most_common(1)[0][0]


def weighted_solution(top_k):

    best = max(top_k, key=lambda x: x["score"])

    solusi = case_solutions.get(best["case_id"], {})

    return {
        "best_case_id": best["case_id"],
        "best_score": best["score"],
        "amar_putusan": solusi.get("amar_putusan", ""),
        "pasal_terbukti": solusi.get("pasal_terbukti", ""),
        "vonis_penjara": solusi.get("vonis_penjara", "")
    } 


def rangkum_via_groq(top_k_hasil: list, query: str) -> str:
    """
    Minta Groq merangkum solusi dari top-k kasus menjadi
    prediksi yang koheren dan mudah dibaca.
    Hanya dipanggil jika GROQ_AKTIF = True.
    """
    if not GROQ_AKTIF or not groq_client:
        return ""

    # Susun konteks dari top-k
    konteks_list = []
    for h in top_k_hasil[:3]:  # batasi 3 kasus agar tidak terlalu panjang
        sol = case_solutions.get(h["case_id"], {})
        konteks_list.append(
            f"Kasus {h['rank']} (skor={h['score']:.3f}):\n"
            f"  Pasal terbukti : {sol.get('pasal_terbukti','-')}\n"
            f"  Amar putusan   : {sol.get('amar_putusan','-')[:200]}\n"
            f"  Vonis penjara  : {sol.get('vonis_penjara','-')}\n"
            f"  Fakta          : {sol.get('ringkasan_fakta','-')[:150]}"
        )
    konteks = "\n\n".join(konteks_list)

    prompt = f"""Kamu adalah asisten analisis hukum perlindungan anak Indonesia.

Berdasarkan {len(top_k_hasil)} kasus serupa berikut, prediksi putusan yang paling mungkin
untuk kasus baru di bawah ini. Jawab dalam 3-4 kalimat: pasal yang kemungkinan terbukti,
jenis putusan (terbukti/bebas/lepas), dan estimasi vonis jika ada.

KASUS SERUPA:
{konteks}

KASUS BARU:
{query[:500]}

Prediksi putusan:"""

    try:
        import time
        resp = groq_client.chat.completions.create(
            model      = CONFIG["GROQ_MODEL"],
            messages   = [{"role": "user", "content": prompt}],
            max_tokens = 300,
            temperature= 0.2,
        )
        time.sleep(CONFIG["GROQ_DELAY"])
        return resp.choices[0].message.content.strip()
    except Exception as e:
        logger.warning(f"Groq rangkum gagal: {e}")
        return ""


print("✅ Fungsi majority_vote_solution, weighted_solution, rangkum_via_groq siap")

✅ Fungsi majority_vote_solution, weighted_solution, rangkum_via_groq siap


## Cell 8 — Fungsi `predict_outcome()` Terpadu

In [10]:
def predict_outcome(query: str, k: int = None, mode: str = None) -> dict:
    """
    Pipeline prediksi solusi (CBR Solution Reuse) untuk satu kasus baru.

    Alur sesuai spesifikasi tugas:
        retrieve top-k → ambil solusi kasus lama → majority vote / weighted

    Returns dict berisi:
        - query              : teks query
        - top_k_cases        : list case_id termirip
        - top_k_scores       : skor similarity
        - majority_solution  : amar putusan yang paling sering muncul di top-k
        - weighted_solution  : solusi dari kasus dengan similarity tertinggi
        - predicted_solution : solusi final (amar putusan)
        - predicted_pasal    : pasal terbukti dari kasus terbaik
        - predicted_vonis    : vonis dari kasus terbaik
    """
    if k    is None: k    = CONFIG["TOP_K"]
    if mode is None: mode = CONFIG["RETRIEVAL_MODE"]

    # Step 1: Retrieve top-k kasus termirip
    top_k = retrieve(query, k=k, mode=mode)
    if not top_k:
        return {"error": "Tidak ada kasus ditemukan"}

    # Step 2a: Majority vote — amar putusan yang paling sering muncul
    majority_sol = majority_vote_solution(top_k)

    # Step 2b: Weighted similarity — solusi dari kasus dengan skor tertinggi
    weighted = weighted_solution(top_k)

    # Step 3: Rangkum via Groq (opsional, jika aktif)
    groq_summary = rangkum_via_groq(top_k, query) if GROQ_AKTIF else ""

    # Step 4: Tentukan solusi final
    # Default pakai weighted (kasus paling mirip), fallback ke majority
    solusi_final = weighted.get("amar_putusan", "") or majority_sol
    if groq_summary:
        solusi_final = groq_summary

    return {
        "query"              : query[:200],
        "mode"               : mode,
        "top_k_cases"        : [h["case_id"] for h in top_k],
        "top_k_scores"       : [h["score"] for h in top_k],
        "majority_solution"  : majority_sol,
        "weighted_solution"  : weighted,
        "groq_summary"       : groq_summary,
        "predicted_solution" : solusi_final,
        "predicted_pasal"    : weighted.get("pasal_terbukti", ""),
        "predicted_vonis"    : weighted.get("vonis_penjara", ""),
        "best_case_id"       : weighted.get("best_case_id", ""),
        "best_score"         : weighted.get("best_score", 0.0),
    }


print("✅ Fungsi predict_outcome() siap (solution-based, tanpa label)")

✅ Fungsi predict_outcome() siap (solution-based, tanpa label)


## Cell 9 — Demo Manual: 5 Kasus Baru

> Sesuaikan teks kasus di bawah dengan domain Perlindungan Anak yang kamu pakai.  
> Idealnya gunakan teks dari putusan yang belum masuk corpus (kasus di luar dataset).

In [13]:
# ============================================================
# 5 KASUS BARU UNTUK DEMO
# Sesuai spesifikasi: bandingkan PREDICTED SOLUTION dgn ACTUAL SOLUTION
# Ganti query_text & amar_sebenarnya dengan kasus nyata jika ada
# ============================================================
KASUS_BARU = [
    {
        "query_id"   : "demo_001",
        "query_text" : (
            "Anak melakukan persetubuhan dengan anak korban berusia 13 tahun. "
            "Perbuatan dilakukan beberapa kali. Didakwa Pasal 81 UU "
            "Perlindungan Anak."
        ),
        "amar_sebenarnya" : "Menjatuhkan pidana penjara dan pelatihan kerja",
        "pasal_sebenarnya": "Pasal 81 UU 35/2014",
    },
    {
        "query_id"   : "demo_002",
        "query_text" : (
            "Anak melakukan pencurian dengan pemberatan bersama temannya "
            "pada malam hari di rumah korban. Didakwa Pasal 363 KUHP."
        ),
        "amar_sebenarnya" : "Menjatuhkan pidana penjara dengan masa percobaan",
        "pasal_sebenarnya": "Pasal 363 KUHP",
    },
    {
        "query_id"   : "demo_003",
        "query_text" : (
            "Anak terbukti memiliki dan menggunakan narkotika golongan I "
            "jenis sabu. Barang bukti berupa kristal bening. Didakwa UU Narkotika."
        ),
        "amar_sebenarnya" : "Menjatuhkan tindakan rehabilitasi / pidana penjara",
        "pasal_sebenarnya": "Pasal 112 UU 35/2009",
    },
    {
        "query_id"   : "demo_004",
        "query_text" : (
            "Perkara anak diselesaikan melalui diversi karena memenuhi syarat. "
            "Tercapai kesepakatan diversi antara para pihak."
        ),
        "amar_sebenarnya" : "Penetapan kesepakatan diversi, perkara dihentikan",
        "pasal_sebenarnya": "Pasal 7 UU SPPA",
    },
    {
        "query_id"   : "demo_005",
        "query_text" : (
            "Anak melakukan penganiayaan terhadap korban hingga luka. "
            "Didakwa Pasal 80 UU Perlindungan Anak."
        ),
        "amar_sebenarnya" : "Menjatuhkan pidana penjara terhadap Anak",
        "pasal_sebenarnya": "Pasal 80 UU 35/2014",
    },
]

# ── Jalankan prediksi ────────────────────────────────────────
print(f"{'='*60}")
print(f"  DEMO PREDIKSI SOLUSI — {len(KASUS_BARU)} KASUS BARU")
print(f"{'='*60}\n")

semua_prediksi = []

for kasus in KASUS_BARU:
    print(f"\n📋 [{kasus['query_id']}]")
    print(f"   Query : {kasus['query_text'][:75]}...")

    hasil = predict_outcome(kasus["query_text"])

    if "error" in hasil:
        print(f"   ⚠️  {hasil['error']}")
        continue

    print(f"   {'─'*50}")
    print(f"   Top-{len(hasil['top_k_cases'])} cases  : {hasil['top_k_cases']}")
    print(f"   Top scores   : {[round(s,3) for s in hasil['top_k_scores']]}")
    print(f"   Best case    : {hasil['best_case_id']} (skor={hasil['best_score']:.3f})")
    print(f"   {'─'*50}")
    print(f"   PREDICTED SOLUTION:")
    print(f"     {hasil['predicted_solution'][:120]}")
    print(f"   PREDICTED PASAL : {hasil['predicted_pasal'][:60]}")
    print(f"   PREDICTED VONIS : {hasil['predicted_vonis'][:60]}")
    print(f"   {'─'*50}")
    print(f"   ACTUAL SOLUTION (referensi):")
    print(f"     {kasus['amar_sebenarnya']}")
    print(f"     {kasus['pasal_sebenarnya']}")

    # Simpan untuk CSV — sesuai spesifikasi: query_id, predicted_solution, top_5_case_ids
    semua_prediksi.append({
        "query_id"           : kasus["query_id"],
        "query_text"         : kasus["query_text"][:200],
        "predicted_solution" : hasil["predicted_solution"][:300],
        "predicted_pasal"    : hasil["predicted_pasal"],
        "predicted_vonis"    : hasil["predicted_vonis"],
        "top_5_case_ids"     : ", ".join(hasil["top_k_cases"]),
        "top_5_scores"       : ", ".join(str(round(s,4)) for s in hasil["top_k_scores"]),
        "best_case_id"       : hasil["best_case_id"],
        "best_score"         : hasil["best_score"],
        "amar_sebenarnya"    : kasus["amar_sebenarnya"],
        "pasal_sebenarnya"   : kasus["pasal_sebenarnya"],
        "groq_summary"       : hasil.get("groq_summary", "")[:300],
    })

print(f"\n{'='*60}")
print(f"  ✅ Selesai — {len(semua_prediksi)} prediksi solusi dihasilkan")
print(f"{'='*60}")

  DEMO PREDIKSI SOLUSI — 5 KASUS BARU


📋 [demo_001]
   Query : Anak melakukan persetubuhan dengan anak korban berusia 13 tahun. Perbuatan ...
   ──────────────────────────────────────────────────
   Top-3 cases  : ['case_053', 'case_097', 'case_073']
   Top scores   : [0.148, 0.137, 0.111]
   Best case    : case_053 (skor=0.148)
   ──────────────────────────────────────────────────
   PREDICTED SOLUTION:
     terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Anak yang Melakukan Persetubuhan Terhadap Anak, seba
   PREDICTED PASAL : 
   PREDICTED VONIS : 3 (tiga) tahun dikurangi selama
   ──────────────────────────────────────────────────
   ACTUAL SOLUTION (referensi):
     Menjatuhkan pidana penjara dan pelatihan kerja
     Pasal 81 UU 35/2014

📋 [demo_002]
   Query : Anak melakukan pencurian dengan pemberatan bersama temannya pada malam hari...
   ──────────────────────────────────────────────────
   Top-3 cases  : ['case_098', 'case_060', 'case_087']
   Top scores 

## Cell 10 — Simpan Hasil Prediksi

In [14]:
# ── Simpan predictions.csv ─────────────────────────────────
# Kolom inti sesuai spesifikasi tugas:
#   query_id | predicted_solution | top_5_case_ids
df_pred = pd.DataFrame(semua_prediksi)
pred_csv = DATA_RESULT / "predictions.csv"
df_pred.to_csv(pred_csv, index=False, encoding="utf-8-sig")
print(f"✅ predictions.csv disimpan: {pred_csv}")
print(f"   Kolom: {list(df_pred.columns)}")

# ── Simpan predictions.json (lengkap) ──────────────────────
pred_json = DATA_RESULT / "predictions.json"
with open(pred_json, "w", encoding="utf-8") as f:
    json.dump(semua_prediksi, f, ensure_ascii=False, indent=2)
print(f"✅ predictions.json disimpan: {pred_json}")

# ── Preview tabel (kolom inti) ──────────────────────────────
print(f"\n📊 Preview predictions.csv (kolom inti):")
kolom_inti = ["query_id", "predicted_solution", "top_5_case_ids"]
print(df_pred[kolom_inti].to_string(index=False))

✅ predictions.csv disimpan: d:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\results\predictions.csv
   Kolom: ['query_id', 'query_text', 'predicted_solution', 'predicted_pasal', 'predicted_vonis', 'top_5_case_ids', 'top_5_scores', 'best_case_id', 'best_score', 'amar_sebenarnya', 'pasal_sebenarnya', 'groq_summary']
✅ predictions.json disimpan: d:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\results\predictions.json

📊 Preview predictions.csv (kolom inti):
query_id                                                                                                                                                                                               predicted_solution               top_5_case_ids
demo_001                       terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Anak yang Melakukan Persetubuhan Terhadap Anak, sebagaimana diatur dan diancam pidana dalam Pasal 81 Ayat (2) UU RI No case_053, case_097, case_073
demo_002 terbukti secara sah dan meyakinkan ber

## Cell 11 — (Opsional) Revisi & Retain

> Tambahkan kasus baru yang terbukti prediksinya benar ke dalam case base.  
> Ini mengimplementasikan siklus **Revise & Retain** dari CBR.

In [15]:
def retain_kasus_baru(hasil_prediksi: dict, kasus_teks: str) -> bool:
    """
    Tambahkan kasus baru ke case base (siklus Retain pada CBR).
    Solusi diambil dari predicted_solution hasil retrieval.

    Args:
        hasil_prediksi : dict hasil predict_outcome()
        kasus_teks     : teks lengkap kasus baru

    Returns: True jika berhasil ditambahkan
    """
    # Load cases saat ini
    with open(DATA_PROC / "cases.json", encoding="utf-8") as f:
        cases_existing = json.load(f)

    new_id = f"case_new_{len(cases_existing)+1:03d}"

    new_case = {
        "case_id"         : new_id,
        "url_detail"      : "",
        "path_txt"        : "",
        "no_perkara"      : "",
        "tanggal_putusan" : "",
        "pengadilan"      : "",
        "terdakwa"        : "",
        "jaksa"           : "",
        "pasal_dakwaan"   : "",
        "pasal_terbukti"  : hasil_prediksi.get("predicted_pasal", ""),
        "amar_putusan"    : hasil_prediksi.get("predicted_solution", ""),
        "vonis_penjara"   : hasil_prediksi.get("predicted_vonis", ""),
        "vonis_denda"     : "",
        "barang_bukti"    : "",
        "ringkasan_fakta" : hasil_prediksi.get("query", "")[:300],
        "jumlah_kata"     : len(kasus_teks.split()),
        "sumber_ekstraksi": "retain",
        "teks_full"       : kasus_teks,
    }

    cases_existing.append(new_case)

    with open(DATA_PROC / "cases.json", "w", encoding="utf-8") as f:
        json.dump(cases_existing, f, ensure_ascii=False, indent=2)

    print(f"✅ Kasus baru {new_id} ditambahkan ke case base")
    print(f"   Total kasus sekarang: {len(cases_existing)}")
    print(f"   ⚠️  Jalankan ulang Tahap 3 untuk update model retrieval")
    return True


# Contoh: retain kasus demo_001 ke case base
if semua_prediksi:
    hasil_001 = predict_outcome(KASUS_BARU[0]["query_text"])
    retain_kasus_baru(hasil_001, KASUS_BARU[0]["query_text"])
else:
    print("⏭️  Tidak ada prediksi untuk di-retain")

✅ Kasus baru case_new_044 ditambahkan ke case base
   Total kasus sekarang: 44
   ⚠️  Jalankan ulang Tahap 3 untuk update model retrieval


---
## ✅ Tahap 4 Selesai!

**Output yang dihasilkan:**
```
/data/results/
├── predictions.csv    ← hasil prediksi semua kasus demo
└── predictions.json   ← versi lengkap
/logs/
└── solution_reuse.log
```

**Struktur `predictions.csv`:**
| Kolom | Keterangan |
|-------|-----------|
| query_id | ID kasus uji |
| label_sebenarnya | Label ground-truth |
| predicted_label | Label hasil prediksi |
| benar | True/False |
| top_5_case_ids | 5 kasus paling mirip |
| majority_vote_label | Prediksi majority vote |
| weighted_sim_label | Prediksi weighted similarity |
| predicted_solution | Teks solusi prediksi |

**Siklus CBR yang sudah selesai:**
```
Retrieve ✅ → Reuse ✅ → Revise ✅ → Retain ✅
```

**Langkah selanjutnya:** Buka `05_evaluation.ipynb` untuk evaluasi lengkap.